# **Notebook 5: Solution V1 — RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [ ]:
!pip install -q chromadb sentence-transformers rouge-score nltk

In [3]:
# YOUR CODE HERE
import json
import os
import zipfile
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Unzip chroma_db if needed
if not os.path.exists("./chroma_db") and os.path.exists("chroma_db.zip"):
  with zipfile.ZipFile("chroma_db.zip", "r") as zf:
    zf.extractall(".")
  print("Extracted chroma_db.zip")

# 2. Load test data and artifacts
df_test = pd.read_csv("df_test.csv")
with open("outputs.json", "r") as f:
  artifacts = json.load(f)

# 3. Connect to ChromaDB
client = chromadb.PersistentClient(path="./chroma_db")
collections = client.list_collections()
collection_name = collections[0].name if collections else "sop_collection"
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = client.get_collection(
    name=collection_name, embedding_function=embed_fn
)

# 4. Load Qwen2.5-1.5B-Instruct on GPU (or CPU fallback)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device.upper()}")

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
  model = model.to("cpu")
model.eval()


def _run_model(messages, max_new_tokens=80):
  prompt = tokenizer.apply_chat_template(
      messages, tokenize=False, add_generation_prompt=True
  )
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
  with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
  generated_ids = outputs[0][inputs["input_ids"].shape[1] :]
  return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def generate_baseline(query):
  messages = [
      {
          "role": "system",
          "content": (
              "You are a helpful customer support assistant. Answer clearly"
              " and concisely."
          ),
      },
      {"role": "user", "content": query},
  ]
  return _run_model(messages)


def generate_naive_rag(query, k=1):
  res = collection.query(query_texts=[query], n_results=k)
  docs = res["documents"][0] if res.get("documents") else []
  metas = res["metadatas"][0] if res.get("metadatas") else []
  context = "\n\n".join(docs)
  source = (
      metas[0].get("source", "sop_doc")
      if metas and isinstance(metas[0], dict)
      else "sop_doc"
  )

  messages = [
      {
          "role": "system",
          "content": (
              "You are a customer support assistant. Answer strictly using the"
              " provided policy context."
          ),
      },
      {
          "role": "user",
          "content": (
              f"Policy Context:\n{context}\n\nCustomer Query: {query}\nAnswer:"
          ),
      },
  ]
  return _run_model(messages), source, context


print("Setup complete!")

Extracted chroma_db.zip


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (16 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Setup complete!


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [4]:
# YOUR CODE HERE
from tqdm.auto import tqdm

SAMPLE_SIZE = 50  # Set to None to evaluate all 391 rows
eval_df = (
    df_test.sample(n=min(SAMPLE_SIZE, len(df_test)), random_state=42).copy()
    if SAMPLE_SIZE is not None
    else df_test.copy()
)

results = []
for idx, row in tqdm(
    eval_df.iterrows(), total=len(eval_df), desc="Evaluating V1 (Baseline + RAG)"
):
  q = row["text"]
  base_ans = generate_baseline(q)
  rag_ans, retrieved_source, retrieved_context = generate_naive_rag(q, k=1)
  results.append({
      "row_idx": idx,
      "query": q,
      "intent": row["intent"],
      "category": row["category"],
      "baseline_output": base_ans,
      "rag_output": rag_ans,
      "retrieved_source": retrieved_source,
      "retrieved_context": retrieved_context,
  })

print(f"Completed automated testing on {len(results)} queries.")

Evaluating V1 (Baseline + RAG):   0%|          | 0/50 [00:00<?, ?it/s]

Completed automated testing on 50 queries.


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [5]:
# YOUR CODE HERE
base_valid = sum(
    1 for r in results if isinstance(r["baseline_output"], str) and r["baseline_output"].strip()
)
rag_valid = sum(
    1 for r in results if isinstance(r["rag_output"], str) and r["rag_output"].strip()
)

total = len(results)
base_adherence = (base_valid / total) * 100 if total else 0.0
rag_adherence = (rag_valid / total) * 100 if total else 0.0

print(f"Baseline Format Adherence:  {base_adherence:.2f}% ({base_valid}/{total})")
print(f"Naive RAG Format Adherence: {rag_adherence:.2f}% ({rag_valid}/{total})")

Baseline Format Adherence:  100.00% (50/50)
Naive RAG Format Adherence: 100.00% (50/50)


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [6]:
# YOUR CODE HERE
import nltk
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
import numpy as np
from rouge_score import rouge_scorer

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
smooth_fn = SmoothingFunction().method1

base_rouge1, base_rougel, base_bleu = [], [], []
rag_rouge1, rag_rougel, rag_bleu = [], [], []

for r in results:
  ref = r.get("retrieved_context", "") or ""
  b_out = r.get("baseline_output", "") or ""
  g_out = r.get("rag_output", "") or ""

  b_scores = scorer.score(ref, b_out)
  g_scores = scorer.score(ref, g_out)

  base_rouge1.append(b_scores["rouge1"].fmeasure)
  base_rougel.append(b_scores["rougeL"].fmeasure)
  rag_rouge1.append(g_scores["rouge1"].fmeasure)
  rag_rougel.append(g_scores["rougeL"].fmeasure)

  ref_tokens = nltk.word_tokenize(ref.lower()) if ref.strip() else [""]
  b_tokens = nltk.word_tokenize(b_out.lower()) if b_out.strip() else [""]
  g_tokens = nltk.word_tokenize(g_out.lower()) if g_out.strip() else [""]

  base_bleu.append(
      sentence_bleu([ref_tokens], b_tokens, smoothing_function=smooth_fn)
  )
  rag_bleu.append(
      sentence_bleu([ref_tokens], g_tokens, smoothing_function=smooth_fn)
  )

  r["base_rouge1"] = b_scores["rouge1"].fmeasure
  r["base_rougel"] = b_scores["rougeL"].fmeasure
  r["base_bleu"] = base_bleu[-1]
  r["rag_rouge1"] = g_scores["rouge1"].fmeasure
  r["rag_rougel"] = g_scores["rougeL"].fmeasure
  r["rag_bleu"] = rag_bleu[-1]

print(f"Baseline  -> ROUGE-1: {np.mean(base_rouge1):.4f} | ROUGE-L: {np.mean(base_rougel):.4f} | BLEU: {np.mean(base_bleu):.4f}")
print(f"Naive RAG -> ROUGE-1: {np.mean(rag_rouge1):.4f} | ROUGE-L: {np.mean(rag_rougel):.4f} | BLEU: {np.mean(rag_bleu):.4f}")

Baseline  -> ROUGE-1: 0.0108 | ROUGE-L: 0.0101 | BLEU: 0.0000
Naive RAG -> ROUGE-1: 0.0346 | ROUGE-L: 0.0274 | BLEU: 0.0000


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [7]:
# YOUR CODE HERE
sample_query = (
    results[0]["query"] if "results" in globals() and results else df_test["text"].iloc[0]
)

base_runs = [generate_baseline(sample_query) for _ in range(3)]
rag_runs = [generate_naive_rag(sample_query, k=1)[0] for _ in range(3)]

base_consistent = len(set(base_runs)) == 1
rag_consistent = len(set(rag_runs)) == 1

assert base_consistent, f"Baseline outputs differed across runs: {base_runs}"
assert rag_consistent, f"Naive RAG outputs differed across runs: {rag_runs}"

consistency_rate = 100.0 if (base_consistent and rag_consistent) else 0.0
print(f"Tested query: {sample_query}")
print(f"Baseline Consistent (3/3 identical): {base_consistent}")
print(f"Naive RAG Consistent (3/3 identical): {rag_consistent}")
print(f"Output Consistency Rate: {consistency_rate:.2f}%")

Tested query: <|im_start|>system
You are a customer support classification assistant. Given a user instruction, respond strictly with a JSON object containing 'intent' and 'category'.<|im_end|>
<|im_start|>user
I cannot afford order {{Order Number}}<|im_end|>
<|im_start|>assistant
{"intent": "cancel_order", "category": "ORDER"}<|im_end|>

Baseline Consistent (3/3 identical): True
Naive RAG Consistent (3/3 identical): True
Output Consistency Rate: 100.00%


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [8]:
# YOUR CODE HERE
import re


def has_unsupported_numbers(answer, sop_context):
  ans_nums = set(re.findall(r"\b\d+(?:\.\d+)?\b", answer or ""))
  sop_nums = set(re.findall(r"\b\d+(?:\.\d+)?\b", sop_context or ""))
  return len(ans_nums - sop_nums) > 0


base_hal, rag_hal = 0, 0
for r in results:
  sop = r.get("retrieved_context", "")
  r["base_hallucinated"] = has_unsupported_numbers(r["baseline_output"], sop)
  r["rag_hallucinated"] = has_unsupported_numbers(r["rag_output"], sop)
  base_hal += int(r["base_hallucinated"])
  rag_hal += int(r["rag_hallucinated"])

total = len(results)
print(
    f"Baseline Hallucination Frequency:  {(base_hal / total) * 100:.2f}%"
    f" ({base_hal}/{total})"
)
print(
    f"Naive RAG Hallucination Frequency: {(rag_hal / total) * 100:.2f}%"
    f" ({rag_hal}/{total})"
)

Baseline Hallucination Frequency:  0.00% (0/50)
Naive RAG Hallucination Frequency: 6.00% (3/50)


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [9]:
# YOUR CODE HERE
import numpy as np
import pandas as pd


def pct_change(base_val, rag_val):
  if base_val == 0:
    return np.nan
  return ((rag_val - base_val) / base_val) * 100


summary_rows = [
    {
        "Metric": "Format Adherence (%)",
        "Baseline": base_adherence,
        "Naive RAG": rag_adherence,
        "Improvement (%)": pct_change(base_adherence, rag_adherence),
    },
    {
        "Metric": "ROUGE-1",
        "Baseline": float(np.mean(base_rouge1)),
        "Naive RAG": float(np.mean(rag_rouge1)),
        "Improvement (%)": pct_change(
            float(np.mean(base_rouge1)), float(np.mean(rag_rouge1))
        ),
    },
    {
        "Metric": "ROUGE-L",
        "Baseline": float(np.mean(base_rougel)),
        "Naive RAG": float(np.mean(rag_rougel)),
        "Improvement (%)": pct_change(
            float(np.mean(base_rougel)), float(np.mean(rag_rougel))
        ),
    },
    {
        "Metric": "BLEU",
        "Baseline": float(np.mean(base_bleu)),
        "Naive RAG": float(np.mean(rag_bleu)),
        "Improvement (%)": pct_change(
            float(np.mean(base_bleu)), float(np.mean(rag_bleu))
        ),
    },
    {
        "Metric": "Output Consistency (%)",
        "Baseline": 100.0 if base_consistent else 0.0,
        "Naive RAG": 100.0 if rag_consistent else 0.0,
        "Improvement (%)": pct_change(
            100.0 if base_consistent else 0.0,
            100.0 if rag_consistent else 0.0,
        ),
    },
    {
        "Metric": "Hallucination Frequency (%)",
        "Baseline": (base_hal / total) * 100 if total else 0.0,
        "Naive RAG": (rag_hal / total) * 100 if total else 0.0,
        "Improvement (%)": pct_change(
            (base_hal / total) * 100 if total else 0.0,
            (rag_hal / total) * 100 if total else 0.0,
        ),
    },
]

comparison_df = pd.DataFrame(summary_rows)
display(comparison_df.round(2))

,Metric,Baseline,Naive RAG,Improvement (%)
0,Format Adherence (%),100.00,100.00,0.00
1,ROUGE-1,0.01,0.03,220.51
2,ROUGE-L,0.01,0.03,170.06
3,BLEU,0.00,0.00,950771.11
4,Output Consistency (%),100.00,100.00,0.00
5,Hallucination Frequency (%),0.00,6.00,NaN


---
## Save Artifacts

In [10]:
# YOUR CODE HERE
import pandas as pd

metrics_df = pd.DataFrame(results)
metrics_df.to_csv("v1_metrics.csv", index=False)
print("Saved v1_metrics.csv with shape:", metrics_df.shape)

Saved v1_metrics.csv with shape: (50, 16)


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**